In [1]:
import pandas as pd
import numpy as np
import scanpy as sc
import anndata as ad
import snapatac2 as snap
import matplotlib.pyplot as plt
import seaborn as sns
from plotnine import *
import pyranges as pr
import os
import warnings
import requests
from importlib import reload
from umap import UMAP
from sklearn.decomposition import PCA
from joblib import Parallel, delayed

from tqdm import tqdm
tqdm.pandas()

import sys
sys.path.append('/scratch/eli')
sys.path.append('/data1/normantm/eli/software')
from perturbseq import *
import multiome as mo

plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Helvetica']
plt.rcParams['font.size'] = 12
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['pdf.fonttype'] = 42

In [2]:
# cellranger calling

id_mtx = sc.read_10x_h5('/data1/normantm/eli/T7/202511_RPE1_dg50/251204_aggr/outs/count/filtered_feature_bc_matrix.h5', gex_only=False)
df = pd.read_csv('/data1/normantm/eli/T7/202511_RPE1_dg50/251204_aggr/outs/count/crispr_analysis/protospacer_calls_per_cell.csv')
df = df.set_index("cell_barcode").reindex(id_mtx.obs_names).reset_index().rename(columns={'index':'barcode'}).fillna(0)

def get_feature(feature_call, pos = 1):
    call = feature_call.split("|")
    for c in call:
        if f"pos{pos}" in c:
            return c.replace(f"_pos{pos}", "")


dfdg = df.query("num_features == 2").copy()
dfdg['target_a'] = dfdg['feature_call'].map(lambda i: get_feature(i, pos = 1))
dfdg['target_b'] = dfdg['feature_call'].map(lambda i: get_feature(i, pos = 2))
dfdg = dfdg.query("target_a.notna() and target_b.notna()").copy()

dfdg['joint_id'] = dfdg.apply(lambda df: df.target_a.split("_")[0] + "_" + df.target_b.split("_")[0], axis=1)
def correct_id(id):
    target_a, target_b = id.split("_")
    if target_a == 'NTC' and target_b == 'NTC':
        return "NTC"
    elif target_a < target_b:
        return f"{target_a}_{target_b}"
    else:
        return f"{target_b}_{target_a}"

dfdg['joint_id_ntc'] = dfdg['joint_id'].apply(correct_id)

/data1/normantm/eli/mamba/miniforge3/envs/scenv/lib/python3.11/site-packages/anndata/_core/anndata.py:1820: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.


In [4]:
# p25 threshold calling

adata = id_mtx[:,id_mtx.var.feature_types == 'CRISPR Guide Capture'].copy()
adata.var_names = adata.var["gene_ids"]
X = adata.X.tocoo()

rows = X.row
cols = X.col
data = X.data

mask = data >= 0

df = pd.DataFrame({
    "barcode": adata.obs_names[rows[mask]].values,
    "feature": adata.var_names[cols[mask]].values,
    "umi_count": data[mask]
})

df['n_guides'] = df.groupby('barcode')['feature'].transform('count')
df['called'] = df.apply(lambda df: df.umi_count >= 5, axis=1)
df['n_called'] = df.groupby('barcode')['called'].transform('sum')
thresholds = dict(df.groupby('feature').umi_count.apply(np.percentile, 25).apply(lambda x: np.maximum(x, 3)))
df['called_threshold'] = df.progress_apply(lambda df: df.umi_count >= thresholds[df['feature']], axis=1)
df['n_called_threshold'] = df.groupby('barcode')['called_threshold'].transform('sum')

ids = df.query("n_called_threshold == 2 and called_threshold").groupby('barcode').feature.agg(lambda x: '|'.join(x)).to_frame().query("feature.str.contains('pos1') and feature.str.contains('pos2')")
ids['threshold_feature_a'] = ids.feature.map(lambda i: get_feature(i, pos = 1))
ids['threshold_feature_b'] = ids.feature.map(lambda i: get_feature(i, pos = 2))
ids = ids.query("threshold_feature_a.notna() and threshold_feature_b.notna()").copy()
ids['threshold_joint_id_ntc'] = ids.apply(lambda df: correct_id(df.threshold_feature_a.split("_")[0] + "_" + df.threshold_feature_b.split("_")[0]), axis=1)

In [5]:
# merge via union

ids_compare = ids[['threshold_feature_a', 'threshold_feature_b', 'threshold_joint_id_ntc']].join(dfdg.set_index('barcode')[['target_a', 'target_b', 'joint_id_ntc']], how = 'outer')
ids_compare['discordant'] = ids_compare.apply(lambda df: df['threshold_joint_id_ntc'] != df['joint_id_ntc'], axis=1)
ids_compare['final_feature_call'] = ids_compare.apply(lambda df: df['joint_id_ntc'] if not pd.isna(df['joint_id_ntc']) else df['threshold_joint_id_ntc'], axis=1)
ids_compare['discordant'].value_counts()

discordant
False    286150
True      32437
Name: count, dtype: int64

In [ ]:
# assign singlets

full_mtx = sc.read_10x_h5('/data1/normantm/eli/T7/202511_RPE1_dg50/251204_aggr/outs/count/filtered_feature_bc_matrix.h5')
assigned = full_mtx[full_mtx.obs_names.isin(ids_compare.index), :].copy()
assigned.obs = assigned.obs.join(ids_compare)

/data1/normantm/eli/mamba/miniforge3/envs/scenv/lib/python3.11/site-packages/anndata/_core/anndata.py:1820: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
/data1/normantm/eli/mamba/miniforge3/envs/scenv/lib/python3.11/site-packages/anndata/_core/anndata.py:1820: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
/data1/normantm/eli/mamba/miniforge3/envs/scenv/lib/python3.11/site-packages/anndata/_core/anndata.py:1820: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.


In [7]:
# filter cells for raw count mtx

assigned.var_names_make_unique()
assigned.var["mito"] = assigned.var_names.str.startswith("MT-")
sc.pp.calculate_qc_metrics(assigned, inplace=True, qc_vars = ["mito"])

def get_guide_target(df):
    
    if df.final_feature_call == 'NTC':
        return 'NTC'
    elif '_NTC' in df.final_feature_call or 'NTC_' in df.final_feature_call:
        return df.final_feature_call.replace('_NTC', '').replace('NTC_', '')
    elif df.final_feature_call != 'NTC' and df.final_feature_call.split("_")[0] == df.final_feature_call.split("_")[1]:
        return df.final_feature_call.split("_")[0]
    else:
        return df.final_feature_call

sc.pp.filter_cells(assigned, min_counts = np.expm1(7.75))
sc.pp.filter_cells(assigned, min_genes = np.expm1(7))
assigned.obs['guide_target'] = assigned.obs.apply(get_guide_target, axis=1)
assigned.obs['is_single'] = assigned.obs.guide_target.map(lambda i: i == 'NTC' or not '_' in i)
assigned.obs['gem_group'] = assigned.obs_names.map(lambda i: i.split('-')[1])
# assigned.write_h5ad("intermediate_files/251204_dg50_gex_counts.h5ad")
assigned

AnnData object with n_obs × n_vars = 316630 × 38606
    obs: 'threshold_feature_a', 'threshold_feature_b', 'threshold_joint_id_ntc', 'target_a', 'target_b', 'joint_id_ntc', 'discordant', 'final_feature_call', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mito', 'log1p_total_counts_mito', 'pct_counts_mito', 'n_counts', 'n_genes', 'guide_target', 'is_single', 'gem_group'
    var: 'gene_ids', 'feature_types', 'genome', 'pattern', 'read', 'sequence', 'mito', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts'

In [ ]:
# filter genes for z-norm mtx

sc.pp.filter_genes(assigned, min_cells = int(len(assigned.obs) * 0.05))
singlets_no_mt = assigned[:,~assigned.var_names.str.startswith("MT-")].copy()

def _get_cellcycle_genes():
    
    dfS = requests.get("https://maayanlab.cloud/Harmonizome/api/1.0/gene_set/S+Phase/Reactome+Pathways+2024").json()
    dfG2M = requests.get("https://maayanlab.cloud/Harmonizome/api/1.0/gene_set/regulation+of+cell+cycle+G2%24slash%24M+phase+transition/GO+Biological+Process+Annotations+2023").json()
    dfM = requests.get("https://maayanlab.cloud/Harmonizome/api/1.0/gene_set/M+Phase/Reactome+Pathways+2024").json()

    S_genes = [e['gene']['symbol'] for e in dfS['associations']]
    G2M_genes = [e['gene']['symbol'] for e in dfG2M['associations']]
    G2M_genes.extend([e['gene']['symbol'] for e in dfM['associations']])
    G2M_genes = list(set(G2M_genes))

    return S_genes, G2M_genes

S_genes, G2M_genes = _get_cellcycle_genes()

with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    
    singlets_no_mt.obs['single_cell'] = True
    cellpop = CellPopulation(
                pd.DataFrame(singlets_no_mt.X.todense().A, index=singlets_no_mt.obs.index, columns=singlets_no_mt.var.index), 
                singlets_no_mt.obs, 
                singlets_no_mt.var, 
                calculate_statistics=False
    )
    
    cellpop.normalized_matrix = normalize_to_gemgroup_control(cellpop, control_cells = "guide_target == 'NTC'")
    cellpop_normalized_sc = ad.AnnData(
                                obs = cellpop.cells, 
                                var = cellpop.genes
                                X = cellpop.where(normalized = True)
    )

valid_mask = ~np.isnan(cellpop_normalized_sc.X).any(axis=0)
cellpop_normalized_sc = cellpop_normalized_sc[:, valid_mask].copy()

sc.tl.pca(cellpop_normalized_sc)
sc.pp.neighbors(cellpop_normalized_sc)
sc.tl.umap(cellpop_normalized_sc)
sc.tl.leiden(cellpop_normalized_sc)
sc.tl.score_genes_cell_cycle(cellpop_normalized_sc, s_genes = S_genes, g2m_genes = G2M_genes)

cellpop_normalized_sc.X = cellpop_normalized_sc.X.astype('float32')
cellpop_normalized_sc.write_h5ad("251204_dg50_gex_raw.h5ad")

In [ ]:
cellpop_regressed = cellpop_normalized_sc.copy()
sc.pp.regress_out(cellpop_regressed, keys = ['pct_counts_mito', 'log1p_total_counts', 'S_score', 'G2M_score'])
sc.tl.pca(cellpop_regressed)
sc.pp.neighbors(cellpop_regressed)
sc.tl.umap(cellpop_regressed)
sc.tl.leiden(cellpop_regressed)
sc.tl.score_genes_cell_cycle(cellpop_regressed, s_genes = S_genes, g2m_genes = G2M_genes)

In [ ]:
import scvi

gex = sc.read('/data1/normantm/eli/T7/202511_RPE1_dg50/analysis/intermediate_files/251204_dg50_gex.h5ad')
raw = sc.read('/data1/normantm/eli/T7/202511_RPE1_dg50/analysis/intermediate_files/251204_dg50_gex_counts.h5ad')
raw = raw[:,raw.var_names.isin(gex.var_names)].copy()
raw.obs = raw.obs.join(gex.obs.phase)
raw.layers['count'] = raw.X
scvi.model.SCVI.setup_anndata(raw, layer = "count", labels_key = 'guide_target', categorical_covariate_keys = ['gem_group', 'phase'])
model = scvi.model.SCVI(raw, n_layers = 3, n_latent = 64)
model.train(
    max_epochs=400,
    batch_size=256,
    early_stopping=True,
    early_stopping_patience=20,
    check_val_every_n_epoch=1,
    enable_progress_bar=True,
    accelerator="auto"
)
model.save('intermediate_files/dg50_scvi')
gex.obsm['X_scvi'] = model.get_latent_representation()

cellpop_regressed.write_h5ad("251204_dg50_gex.h5ad")